# Day 2: Ingest CSV into Bronze using COPY INTO
Requirement: Ingest CSV (transactions) into Bronze (Delta) Using Copy INTO Command
Apply add audit columns (load_dt, source).
Create and add descriptions/metadata about enterprise data to make it more discoverable.

In [0]:
# Define paths and table names
source_path = "/Volumes/vstone/bronze/raw_volume/csv_initial/"
catalog = "vstone"
schema = "bronze"
bronze_table_name = "flight_bronze"
bronze_table_full_name = f"{catalog}.{schema}.{bronze_table_name}"

### 1. Create the Bronze Table (Empty if not exists)
We will infer the schema initially or define it explicitly. Since the data is flight delays, we define a generic schema.
To use COPY INTO effectively with audit columns, we often create an empty table first or let Databricks infer it (if supported in the specific DBR version).

In [0]:
%sql
show tables in vstone.bronze

In [0]:
from pyspark.sql.functions import current_timestamp, lit

if not spark.catalog.tableExists(bronze_table_full_name):

    print("Creating Bronze table...")

    sample_df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(source_path)
        .limit(0)
    )

    empty_df = (
        sample_df
        .withColumn("load_dt", current_timestamp())
        .withColumn("source", lit(source_path))
    )

    empty_df.write \
        .format("delta") \
        .saveAsTable(bronze_table_full_name)

    print("Table created")

else:
    print("Table already exists")

print("Table Ready")

### 2. Run COPY INTO command
`COPY INTO` is idempotent and handles incremental loads efficiently.

In [0]:
%sql
-- We use spark.sql to run this dynamically since we defined variables in Python
-- However, for demonstration, here is the direct SQL:
-- COPY INTO main.default.bronze_flight_delays_csv
-- FROM '/Volumes/main/default/raw_data/chunk1_csv_initial/'
-- FILEFORMAT = CSV
-- FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true')
-- COPY_OPTIONS ('mergeSchema' = 'true')

In [0]:
copy_into_query = f"""
COPY INTO {bronze_table_full_name}
FROM '{source_path}'
FILEFORMAT = CSV
FORMAT_OPTIONS (
    'header' = 'true',
    'inferSchema' = 'true'
)
COPY_OPTIONS (
    'mergeSchema' = 'true'
)
"""

print("Executing COPY INTO...")
display(spark.sql(copy_into_query))

In [0]:
%sql
SELECT COUNT(*) FROM vstone.bronze.flight_bronze;

In [0]:
%sql
DESCRIBE vstone.bronze.flight_bronze;

In [0]:
%sql
select * from vstone.bronze.flight_bronze
limit 10;

### 3. Add Metadata and Descriptions
Making data discoverable using Unity Catalog comments.

In [0]:
# Add table description
spark.sql(f"COMMENT ON TABLE {bronze_table_full_name} IS 'Bronze layer table for Flight Delays ingested from CSV. Contains raw data with audit columns.'")

# Add column descriptions (Example)
try:
    spark.sql(f"ALTER TABLE {bronze_table_full_name} CHANGE COLUMN load_dt COMMENT 'Timestamp when the record was ingested into Bronze'")
    spark.sql(f"ALTER TABLE {bronze_table_full_name} CHANGE COLUMN source COMMENT 'Source file path in Volumes'")
except Exception as e:
    print(f"Note: Column comments might require specific Unity Catalog permissions. {e}")

print("Day 2: COPY INTO ingestion completed successfully.")